In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import os

In [10]:
def get_latest_xp_renda_fixa_pdf():
    download_dir = os.getcwd()
    
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Configurar preferências
    prefs = {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=chrome_options)
    
    # Habilitar download em modo headless via CDP
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": download_dir
    })
    
    try:
        base_url = "https://conteudos.xpi.com.br/a-semana-na-renda-fixa/"
        print(f"Acessando a página principal: {base_url}")
        driver.get(base_url)
        
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/relatorios/']"))
        )
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        latest_article_link = None
        for a in soup.find_all('a', href=True):
            if "a-semana-na-renda-fixa" in a['href'] and "/relatorios/" in a['href']:
                latest_article_link = a['href']
                break

        if not latest_article_link:
            print("Não foi possível encontrar o link do artigo mais recente.")
            return

        print(f"Artigo mais recente encontrado: {latest_article_link}")
        driver.get(latest_article_link)
        
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "a"))
        )
        
        print("Procurando link de download...")
        links = driver.find_elements(By.TAG_NAME, "a")
        download_element = None
        
        for link in links:
            text = link.text.lower()
            href = link.get_attribute("href")
            if href and href.endswith(".pdf"):
                if "confira aqui o relatório" in text or "baixar" in text:
                    download_element = link
                    break
        
        if not download_element:
            for link in links:
                href = link.get_attribute("href")
                if href and href.endswith(".pdf"):
                    download_element = link
                    break
        
        if download_element:
            pdf_url = download_element.get_attribute("href")
            filename = pdf_url.split('/')[-1]
            print(f"Iniciando download via clique (CDP): {filename}")
            
            # Clicar usando JavaScript para evitar problemas de visibilidade
            driver.execute_script("arguments[0].click();", download_element)
            
            # Aguardar o download
            timeout = 30
            start_time = time.time()
            print("Aguardando conclusão do download...")
            
            while time.time() - start_time < timeout:
                files = os.listdir(download_dir)
                pdf_files = [f for f in files if f.endswith('.pdf') and not f.endswith('.crdownload')]
                if pdf_files:
                    # Ordenar por data de modificação para pegar o mais recente se houver vários
                    pdf_files.sort(key=lambda x: os.path.getmtime(os.path.join(download_dir, x)), reverse=True)
                    print(f"Download concluído com sucesso! Arquivo: {pdf_files[0]}")
                    return pdf_files[0]
                time.sleep(1)
            
            print("Tempo limite atingido.")
        else:
            print("Link de download não encontrado.")
            
    finally:
        driver.quit()

if __name__ == "__main__":
    get_latest_xp_renda_fixa_pdf()

Acessando a página principal: https://conteudos.xpi.com.br/a-semana-na-renda-fixa/
Artigo mais recente encontrado: https://conteudos.xpi.com.br/renda-fixa/relatorios/a-semana-na-renda-fixa-23-02-26-a-27-02-26/
Procurando link de download...
Iniciando download via clique (CDP): A-Semana-na-Renda-Fixa-23-27-fev26-XP-Research.pdf
Aguardando conclusão do download...
Download concluído com sucesso! Arquivo: A-Semana-na-Renda-Fixa-16-20-fev26-XP-Research-1.pdf


In [11]:
# Importar bibliotecas para processar PDF
import fitz  # PyMuPDF
import pdfplumber
import shutil

In [12]:
def extract_pdf_content(pdf_path):
    """
    Extrai texto, tabelas, imagens embutidas e graficos vetoriais renderizados de um PDF.

    Args:
        pdf_path: caminho do arquivo PDF

    Returns:
        dict: dados extraidos (texto, tabelas, imagens e graficos)
    """

    output_dir = "pdf_extraidos"
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # ========== EXTRAIR TEXTO E TABELAS COM PDFPLUMBER ==========
    texto_completo = ""
    tabelas_encontradas = []

    print(f"\nProcessando arquivo: {pdf_path}")

    with pdfplumber.open(pdf_path) as pdf:
        num_pages = len(pdf.pages)
        print(f"Total de paginas: {num_pages}\n")

        for i, page in enumerate(pdf.pages, 1):
            texto_pagina = page.extract_text()
            if texto_pagina:
                texto_completo += f"\n{'='*80}\nPAGINA {i}\n{'='*80}\n{texto_pagina}"

            tabelas = page.extract_tables()
            if tabelas:
                for j, tabela in enumerate(tabelas, 1):
                    tabelas_encontradas.append({
                        'pagina': i,
                        'numero': j,
                        'dados': tabela
                    })
                    print(f"Tabela encontrada na pagina {i}")

    # ========== EXTRAIR IMAGENS E GRAFICOS COM PYMUPDF ==========
    imagens_salvas = []
    graficos_salvos = []
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, 1):
        imagens = page.get_images(full=True)

        # 1) Imagens embutidas no PDF
        for img_index, img in enumerate(imagens, 1):
            try:
                xref = img[0]
                pix = fitz.Pixmap(doc, xref)

                if pix.n - pix.alpha >= 4:
                    pix = fitz.Pixmap(fitz.csRGB, pix)

                imagem_filename = f"{output_dir}/imagem_embutida_pag{page_num}_{img_index}.png"
                pix.save(imagem_filename)
                imagens_salvas.append(imagem_filename)
                print(f"Imagem embutida salva: {imagem_filename}")
            except Exception as e:
                print(f"Erro ao salvar imagem embutida da pagina {page_num}: {e}")

        # 2) Render da pagina inteira para capturar conteudo vetorial (graficos, linhas, etc.)
        try:
            zoom = 2.0
            mat = fitz.Matrix(zoom, zoom)
            pagina_render = f"{output_dir}/pagina_renderizada_{page_num}.png"
            pix_page = page.get_pixmap(matrix=mat, alpha=False)
            pix_page.save(pagina_render)
            graficos_salvos.append(pagina_render)
            print(f"Pagina renderizada salva: {pagina_render}")
        except Exception as e:
            print(f"Erro ao renderizar pagina {page_num}: {e}")

        # 3) Tenta recortar regioes de figura quando encontra legenda
        try:
            termos_legenda = ["Figura", "CURVA DI FUTURO", "Curva DI Futuro"]
            areas_legenda = []
            for termo in termos_legenda:
                areas_legenda.extend(page.search_for(termo))

            if areas_legenda:
                for idx, area in enumerate(areas_legenda, 1):
                    y_inicio = max(0, area.y0 - 20)
                    y_fim = min(page.rect.height, area.y1 + page.rect.height * 0.50)
                    clip = fitz.Rect(0, y_inicio, page.rect.width, y_fim)

                    grafico_file = f"{output_dir}/grafico_pag{page_num}_{idx}.png"
                    pix_clip = page.get_pixmap(matrix=fitz.Matrix(3, 3), clip=clip, alpha=False)
                    pix_clip.save(grafico_file)
                    graficos_salvos.append(grafico_file)
                    print(f"Recorte de grafico salvo: {grafico_file}")
        except Exception as e:
            print(f"Erro ao gerar recorte de grafico na pagina {page_num}: {e}")

    doc.close()

    # ========== SALVAR TEXTO EM ARQUIVO ==========
    if texto_completo:
        texto_filename = "texto_extraido.txt"
        with open(texto_filename, 'w', encoding='utf-8') as f:
            f.write(texto_completo)
        print(f"\nTexto salvo em: {texto_filename}")

    resultado = {
        'texto': texto_completo,
        'imagens': imagens_salvas,
        'graficos': graficos_salvos,
        'tabelas': tabelas_encontradas,
        'total_imagens': len(imagens_salvas),
        'total_graficos': len(graficos_salvos),
        'total_tabelas': len(tabelas_encontradas)
    }

    print("\nResumo da Extracao:")
    print(f"   - Imagens embutidas: {resultado['total_imagens']}")
    print(f"   - Graficos/renderizacoes: {resultado['total_graficos']}")
    print(f"   - Tabelas encontradas: {resultado['total_tabelas']}")
    print(f"   - Caracteres de texto: {len(texto_completo)}\n")

    return resultado

In [13]:
# ========== EXECUTAR O FLUXO COMPLETO ==========
if __name__ == "__main__":
    # Passo 1: Baixar o PDF mais recente
    print("=" * 80)
    print("INICIANDO AUTOMACAO DE DOWNLOAD E EXTRACAO DE PDF")
    print("=" * 80)

    pdf_filename = get_latest_xp_renda_fixa_pdf()

    if pdf_filename:
        # Passo 2: Processar e extrair conteudo do PDF
        pdf_path = os.path.join(os.getcwd(), pdf_filename)
        resultado = extract_pdf_content(pdf_path)

        print("\n" + "=" * 80)
        print("DADOS EXTRAIDOS - PRIMEIROS 500 CARACTERES:")
        print("=" * 80)
        print(resultado['texto'][:500] + "...\n")

        if resultado['tabelas']:
            print("TABELAS ENCONTRADAS:")
            for tabela_info in resultado['tabelas']:
                print(f"  - Pagina {tabela_info['pagina']}, Tabela #{tabela_info['numero']}")

        if resultado['imagens']:
            print(f"\nIMAGENS EMBUTIDAS EXTRAIDAS ({len(resultado['imagens'])} total):")
            for img in resultado['imagens']:
                print(f"  - {img}")

        if resultado['graficos']:
            print(f"\nGRAFICOS/RENDERS EXTRAIDOS ({len(resultado['graficos'])} total):")
            for graf in resultado['graficos'][:10]:
                print(f"  - {graf}")

            if len(resultado['graficos']) > 10:
                print(f"  ... e mais {len(resultado['graficos']) - 10} arquivo(s)")
    else:
        print("Nao foi possivel baixar o PDF")

INICIANDO AUTOMACAO DE DOWNLOAD E EXTRACAO DE PDF
Acessando a página principal: https://conteudos.xpi.com.br/a-semana-na-renda-fixa/
Artigo mais recente encontrado: https://conteudos.xpi.com.br/renda-fixa/relatorios/a-semana-na-renda-fixa-23-02-26-a-27-02-26/
Procurando link de download...
Iniciando download via clique (CDP): A-Semana-na-Renda-Fixa-23-27-fev26-XP-Research.pdf
Aguardando conclusão do download...
Download concluído com sucesso! Arquivo: A-Semana-na-Renda-Fixa-16-20-fev26-XP-Research-1.pdf

Processando arquivo: c:\Users\gabriel.condezin\automate\A-Semana-na-Renda-Fixa-16-20-fev26-XP-Research-1.pdf
Total de paginas: 28

Tabela encontrada na pagina 1
Tabela encontrada na pagina 1
Tabela encontrada na pagina 1
Tabela encontrada na pagina 2
Tabela encontrada na pagina 2
Tabela encontrada na pagina 3
Tabela encontrada na pagina 3
Tabela encontrada na pagina 4
Tabela encontrada na pagina 4
Tabela encontrada na pagina 4
Tabela encontrada na pagina 4
Tabela encontrada na pagina 4